# Ridge Regression Using Batch Gradient Descent

In this notebook, we implement **Ridge Regression from scratch** using the **Batch Gradient Descent (BGD)** optimization algorithm.

The implementation includes:

- Loading the dataset
- Separating features and target
- Train-test splitting
- Feature scaling
- Ridge regularization
- Batch Gradient Descent
- Model prediction
- Intercept and coefficient extraction
- R² score evaluation

## 1. Import Required Libraries

We import the required Python libraries:

- **NumPy** for numerical operations
- **Pandas** for loading and manipulating the dataset
- **train_test_split** for splitting the dataset into training and testing sets
- **StandardScaler** for feature standardization
- **r2_score** for evaluating the regression model

In [12]:
import numpy as np
import pandas as pd

## 2. Load the Dataset

The diabetes dataset is loaded using Pandas.

The dataset contains several input features and one target variable named **Target**.

The target variable will be predicted using Ridge Regression.

In [13]:
df=pd.read_csv(r"C:\Users\Niloy\Downloads\ml practice\ML Algorithms\Diabetes.csv")

## 3. Separate Features and Target

We divide the dataset into two parts:

- **X** → Independent variables/features
- **y** → Dependent variable/target

The `Target` column is removed from `X` because it is the value that our model needs to predict.

In [14]:
X=df.drop('Target',axis=1)
y=df["Target"]

## 4. Train-Test Split

The dataset is divided into:

- **80% training data**
- **20% testing data**

The training data is used to learn the model parameters, while the testing data is used to evaluate how well the trained model performs on unseen data.

`random_state=42` is used to make the split reproducible.

In [15]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 5. Feature Scaling

Before applying Gradient Descent, the features are standardized using `StandardScaler`.

Standardization transforms each feature approximately into:

- Mean = 0
- Standard deviation = 1

This is important for Gradient Descent because features with very different scales can make optimization slower or unstable.

The scaler is **fitted only on the training data** and then used to transform both the training and testing data.

In [16]:
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

## 6. Ridge Regression

Ridge Regression is a linear regression technique that adds an **L2 regularization penalty** to the loss function.

The objective function is:

$$
J(\beta)=\frac{1}{2m}\sum_{i=1}^{m}(y_i-\hat{y}_i)^2
+\frac{\alpha}{2m}\sum_{j=1}^{n}\beta_j^2
$$

where:

- $m$ = number of training samples
- $n$ = number of features
- $y_i$ = actual target value
- $\hat{y}_i$ = predicted target value
- $\beta_j$ = coefficient of feature $j$
- $\alpha$ = regularization strength

A larger value of $\alpha$ applies stronger regularization and pushes the coefficients closer to zero.

The intercept is **not penalized** in Ridge Regression.

## 7. Batch Gradient Descent

Batch Gradient Descent calculates the gradient using the **entire training dataset** in every iteration.

For the Mean Squared Error part, the gradient is:

$$
\nabla J(\beta)=\frac{1}{m}X^T(X\beta-y)
$$

For Ridge Regression, the regularization term is added:

$$
\nabla J(\beta)
=
\frac{1}{m}X^T(X\beta-y)
+
\frac{\alpha}{m}P\beta
$$

where $P$ is the penalty matrix.

The intercept is not regularized, so the first diagonal element of $P$ is set to zero.

The parameters are updated using:

$$
\beta := \beta-\eta\nabla J(\beta)
$$

where:

- $\beta$ = model parameters
- $\eta$ = learning rate
- $\nabla J(\beta)$ = gradient of the cost function

In [17]:
class RidgeBatchGD:
    def __init__(self,alpha,learning_rate,epochs):
        self.alpha = alpha
        self.lr = learning_rate
        self.epoch = epochs

        self.beta = None
        self.intercept = None
        self.co_efficient = None

    def fit(self,X_train,y_train):
        # m = samples  , n= features
        m,n = X_train.shape

        X_train = np.asarray(X_train,dtype=float)
        y_train = np.asarray(y_train,dtype=float)

        X_train = np.insert(X_train,0,1,axis=1)

        self.beta = np.zeros(n+1)

        #ridge penalty matrix
        p=np.eye(n+1)
        p[0,0]=0

        for i in range(self.epoch):
            y_pred = X_train@self.beta
            error = y_pred - y_train
            # gradient of MSE
            gradient = (X_train.T@error)/m 
            gradient += (self.alpha/m)*(p@self.beta)

            self.beta-= self.lr*gradient

        self.intercept = self.beta[0]
        self.co_efficient = self.beta[1:]

    def predict(self,X_test):
        X_test = np.asarray(X_test,dtype=float)
        return X_test@self.co_efficient + self.intercept
    def intercept(self):
        return self.intercept
    def co_efficient(self):
        return self.co_efficient

## 8. Implement Ridge Regression Using Batch Gradient Descent

The `RidgeBatchGD` class implements Ridge Regression from scratch.

The class contains:

- `__init__()` → initializes hyperparameters
- `fit()` → trains the model using Batch Gradient Descent
- `predict()` → generates predictions
- `intercept` → stores the learned intercept
- `co_efficient` → stores the learned feature coefficients

The main hyperparameters are:

- `alpha` → Ridge regularization strength
- `learning_rate` → step size used during gradient descent
- `epochs` → number of gradient descent iterations

## 9. Initialize Model Parameters

Inside the constructor, the following parameters are initialized:

### `alpha`

Controls the strength of L2 regularization.

### `learning_rate`

Controls how large each gradient descent update is.

### `epochs`

Determines how many times the complete training dataset is used to update the parameters.

The model parameters are initially set to `None` because they will be learned during training.

## 10. Training the Model

The `fit()` method trains the Ridge Regression model.

First, the number of samples and features are obtained:

- `m` → number of samples
- `n` → number of features

The input data is converted into NumPy arrays so that matrix operations can be performed efficiently.

## 11. Add the Intercept Column

A column of ones is added to the training feature matrix.

This allows the model to represent the intercept as the first parameter of the parameter vector.

Therefore:

$$
X =
\begin{bmatrix}
1 & x_{11} & x_{12} & \cdots & x_{1n}\\
1 & x_{21} & x_{22} & \cdots & x_{2n}\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
1 & x_{m1} & x_{m2} & \cdots & x_{mn}
\end{bmatrix}
$$

and

$$
\beta =
\begin{bmatrix}
\beta_0\\
\beta_1\\
\vdots\\
\beta_n
\end{bmatrix}
$$

Here, $\beta_0$ represents the intercept.

## 12. Initialize Model Coefficients

All model parameters are initially set to zero:

$$
\beta =
[0,0,\ldots,0]
$$

The vector contains:

- One parameter for the intercept
- One parameter for each feature

These values are updated during Gradient Descent.

## 13. Create the Ridge Penalty Matrix

An identity matrix is used to implement L2 regularization.

However, the intercept should not be penalized.

Therefore, the first diagonal element is changed to zero:

$$
P=
\begin{bmatrix}
0&0&0&\cdots&0\\
0&1&0&\cdots&0\\
0&0&1&\cdots&0\\
\vdots&\vdots&\vdots&\ddots&\vdots\\
0&0&0&\cdots&1
\end{bmatrix}
$$

This ensures that the Ridge penalty is applied only to the feature coefficients, not to the intercept.

## 14. Calculate Predictions During Training

At every epoch, the current model parameters are used to calculate predictions:

$$
\hat{y}=X\beta
$$

The prediction error is then calculated as:

$$
error=\hat{y}-y
$$

This error is used to calculate the gradient.

## 15. Calculate the Gradient

The gradient of the Mean Squared Error component is calculated using:

$$
\frac{1}{m}X^T(\hat{y}-y)
$$

The Ridge regularization gradient is:

$$
\frac{\alpha}{m}P\beta
$$

Therefore, the complete Ridge Gradient is:

$$
\nabla J(\beta)
=
\frac{1}{m}X^T(\hat{y}-y)
+
\frac{\alpha}{m}P\beta
$$

Both terms are combined before updating the model parameters.

## 16. Update the Model Parameters

The parameters are updated using the Batch Gradient Descent rule:

$$
\beta_{new}
=
\beta_{old}
-
\eta\nabla J(\beta)
$$

The process is repeated for the specified number of epochs.

Since Batch Gradient Descent uses the complete training dataset for every update, each iteration uses all training samples.

## 17. Store the Learned Parameters

After training is completed:

- The first element of `beta` is stored as the **intercept**
- The remaining elements are stored as the **feature coefficients**

Therefore:

$$
\beta_0 = \text{intercept}
$$

and

$$
[\beta_1,\beta_2,\ldots,\beta_n]
=
\text{feature coefficients}
$$

## 18. Make Predictions

The `predict()` method uses the learned coefficients and intercept to predict the target values for new data.

The prediction equation is:

$$
\hat{y}
=
X\beta+b
$$

where:

- $X$ = input features
- $\beta$ = learned coefficients
- $b$ = learned intercept
- $\hat{y}$ = predicted target

## 19. Create the Ridge Regression Model

The Ridge Regression model is initialized with:

- `alpha = 0.1`
- `learning_rate = 0.01`
- `epochs = 1000`

These values determine the regularization strength, gradient descent step size, and number of training iterations.

In [18]:
RBGD = RidgeBatchGD(0.1,0.01,1000)

## 20. Train the Model

The model is trained using the scaled training data.

During training, Batch Gradient Descent repeatedly calculates the prediction error and updates the model coefficients to minimize the Ridge Regression objective function.

In [19]:
RBGD.fit(X_train_scaled,y_train)

## 21. Predict the Test Set

After training, the learned model is used to predict the target values of the unseen test dataset.

The resulting predictions are stored in `y_pred`.

In [20]:
y_pred = RBGD.predict(X_test_scaled)
y_pred

array([705.36008329, 761.19967943, 720.16149305, 865.30927572,
       505.28974222, 764.57884558, 743.4028955 , 583.14553613,
       819.4215989 , 927.93057113, 589.29760719, 670.4900256 ,
       744.5224962 , 913.46917   , 649.81919115, 792.81377797,
       565.49698173, 733.92193407, 587.24055532, 534.76190881,
       798.63626234, 525.08393496, 772.675237  , 569.33412017,
       608.1156753 , 597.54436446, 823.03809804, 712.77069141,
       795.78462273, 774.17655089, 666.83560307, 820.71776388,
       631.10295362, 928.844339  , 630.36478637, 664.13061356,
       833.87031918, 763.42996372, 560.52818614, 562.69558822])

## 22. Display Intercept and Coefficients

The learned model parameters are displayed.

### Intercept

The intercept represents the predicted target value when all standardized input features are zero.

### Coefficients

Each coefficient represents the contribution of its corresponding standardized feature to the predicted target.

Because Ridge Regression applies L2 regularization, the coefficients are generally shrunk toward zero.

In [23]:
print("Intercept : ", RBGD.intercept)
print("Co-efficient : ", RBGD.co_efficient)

Intercept :  696.6949215126479
Co-efficient :  [18.14880338 22.50726988 54.85439154 68.98008944]


## 23. Evaluate the Model Using R² Score

The model is evaluated using the coefficient of determination, or **R² score**.

The formula is:

$$
R^2
=
1-
\frac{\sum(y_i-\hat{y}_i)^2}
{\sum(y_i-\bar{y})^2}
$$

where:

- $y_i$ = actual target
- $\hat{y}_i$ = predicted target
- $\bar{y}$ = mean of the actual target values

### Interpretation

- $R^2 = 1$ → perfect prediction
- $R^2 = 0$ → model performs approximately like predicting the mean
- $R^2 < 0$ → model performs worse than simply predicting the mean

A higher R² value generally indicates a better fit to the test data.

In [24]:
from sklearn.metrics import r2_score
r2=r2_score(y_test,y_pred)
r2

0.9477749243972877

## 24. Summary

In this notebook, Ridge Regression was implemented **from scratch using Batch Gradient Descent**.

The complete workflow was:

1. Load the dataset
2. Separate features and target
3. Split the data into training and testing sets
4. Standardize the features
5. Add an intercept term
6. Initialize the model parameters
7. Create the Ridge penalty matrix
8. Calculate predictions
9. Calculate the MSE gradient
10. Calculate the Ridge regularization gradient
11. Update parameters using Batch Gradient Descent
12. Store the intercept and coefficients
13. Predict the test data
14. Evaluate the model using R² score

This implementation demonstrates how Ridge Regression works internally instead of relying on the ready-made `sklearn.linear_model.Ridge` implementation.